# AllSortsHub Cartoon Studio — Episode 1 Colab Generator

**T4-safe mode:** uses Stable Video Diffusion XT 1.1 image-to-video instead of the RAM-heavy Wan/CogVideoX paths. It renders short clips, checkpoints each clip to Google Drive, and resumes after disconnects.

**Hugging Face access:** SVD-XT 1.1 is a gated Stability AI model. Before running cell 4, accept the model terms on Hugging Face and add a Hugging Face access token to Colab Secrets as `HF_TOKEN`.

In [ ]:
# 1. GPU check
!nvidia-smi
import torch, shutil
if not torch.cuda.is_available(): raise RuntimeError('No CUDA GPU. Choose Runtime > Change runtime type > GPU.')
GPU_NAME=torch.cuda.get_device_name(0); VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3
print('PyTorch:',torch.__version__); print('GPU:',GPU_NAME); print('VRAM:',round(VRAM_GB,1),'GB'); print('FFmpeg:',shutil.which('ffmpeg'))
if VRAM_GB < 8: raise RuntimeError('At least 8 GB VRAM is required.')
USE_T4_PATH=VRAM_GB < 20
if not USE_T4_PATH: raise RuntimeError('This notebook revision is intentionally T4-focused. Use the Wan 2.2 24GB+ notebook on a larger GPU.')
print('Generation path: Stable Video Diffusion XT 1.1 T4-safe')

In [ ]:
# 2. Persistent Drive + project
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/AllSortsHub-Wan2.2'
!mkdir -p "$BASE/models" "$BASE/generated" "$BASE/output"
%cd /content
!rm -rf cartoon-studio
!git clone -q https://github.com/parth01/AllSortsHub-Cartoon-Studio.git cartoon-studio
!python -m pip install -q -U 'diffusers>=0.30.0,<0.36.0' transformers accelerate safetensors huggingface_hub
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
# 3. Configure low-RAM image-to-video generation
MODEL_ID='stabilityai/stable-video-diffusion-img2vid-xt-1-1'
VID_WIDTH,VID_HEIGHT,VID_FRAMES,VID_STEPS=576,320,14,20
print(f'SVD-XT 1.1: {VID_WIDTH}x{VID_HEIGHT}, {VID_FRAMES} frames, {VID_STEPS} steps')

In [ ]:
# 4. Authenticate to Hugging Face and load SVD-XT
import torch, gc
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import export_to_video, load_image

# SVD-XT 1.1 is a gated Hugging Face model.
# Store your token in Colab: left sidebar -> Secrets (key icon) -> add HF_TOKEN.
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN=None

if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN is missing. First accept the SVD-XT 1.1 model terms on Hugging Face, '
        'create a read access token, then add it to Colab Secrets as HF_TOKEN and rerun this cell.'
    )

print('Loading Stable Video Diffusion XT 1.1...')
pipe=StableVideoDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant='fp16', token=HF_TOKEN)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
print('SVD-XT pipeline ready.')

In [ ]:
# 5. Prepare Episode 1 and restore checkpoints
from pathlib import Path
import json, shutil
ROOT=Path('/content/cartoon-studio/master-version/AllSortsHub-Billion 2'); LOCAL_GEN=ROOT/'wan_i2v'/'generated'; DRIVE_GEN=Path(BASE)/'generated'
LOCAL_GEN.mkdir(parents=True,exist_ok=True); DRIVE_GEN.mkdir(parents=True,exist_ok=True)
manifest=json.loads((ROOT/'wan_i2v'/'manifest.json').read_text())
for p in DRIVE_GEN.glob('shot_*.mp4'):
    t=LOCAL_GEN/p.name
    if not t.exists() or t.stat().st_size<10000: shutil.copy2(p,t)
print('Episode shots:',len(manifest['shots']))
print('Restored clips:',len(list(LOCAL_GEN.glob('shot_*.mp4'))))

In [ ]:
# 6. Resumable Episode 1 generation
import gc, shutil, torch
from PIL import Image
from diffusers.utils import export_to_video
prompts=(ROOT/'wan_i2v'/'prompts.txt').read_text()
STYLE='Modern 2D cel-shaded cartoon animation, bold clean black linework, semi-flat shading, vibrant colors, expressive facial acting, preserve the exact character designs and environment in the input image. Smooth readable hand-drawn motion. Keep faces, hair, clothing, proportions, props and background layout consistent.'
NEG='No photorealism, no 3D CGI, no live action, no extra fingers, no duplicate limbs, no warped faces, no character morphing, no costume changes, no hairstyle changes, no background replacement, no random objects, no random text, no logos, no watermark, no scene cuts, no extreme deformation.'
def prompt_for(n):
    marker=f'SHOT {n:02d} —'; s=prompts.find(marker); e=prompts.find('\n\nSHOT ',s+2); e=prompts.find('\n\nNEGATIVE',s+2) if e<0 else e
    if s<0: raise RuntimeError('Missing prompt for '+marker)
    return f'{STYLE} {prompts[s:e if e>=0 else None].split(chr(10),1)[1].strip()} {NEG}'
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
def run_svd(image,prompt,out,seed):
    if out.exists() and out.stat().st_size>10000: print('SKIP',out.name); return
    print('GENERATING',out.name,f'({VID_WIDTH}x{VID_HEIGHT}, {VID_FRAMES} frames, {VID_STEPS} steps)')
    cleanup_gpu()
    img=Image.open(image).convert('RGB').resize((VID_WIDTH,VID_HEIGHT))
    gen=torch.Generator(device='cuda').manual_seed(seed)
    try:
        result=pipe(image=img, height=VID_HEIGHT, width=VID_WIDTH, num_frames=VID_FRAMES, num_inference_steps=VID_STEPS, min_guidance_scale=1.0, max_guidance_scale=2.5, motion_bucket_id=127, noise_aug_strength=0.02, generator=gen)
        export_to_video(result.frames[0],str(out),fps=7)
    except Exception as e:
        cleanup_gpu(); print('SVD generation error:',repr(e)); raise
    cleanup_gpu()
for shot in manifest['shots']:
    n=int(shot['id']); image=ROOT/shot['image']; p=prompt_for(n); out=LOCAL_GEN/f'shot_{n:02d}.mp4'
    run_svd(image,p,out,910000+n)
    if out.exists() and out.stat().st_size>10000: shutil.copy2(out,DRIVE_GEN/out.name); print('CHECKPOINT',out.name)
    cleanup_gpu()
print('Generation pass complete.')

In [ ]:
# 7. Assemble final Episode 1
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!python3 wan_i2v/assemble_episode.py
!cp -f output/AllSortsHub_Episode_01_WAN_MASTER.mp4 "$BASE/output/"
!cp -f output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4 "$BASE/output/"
!ls -lh output/AllSortsHub_Episode_01_WAN_MASTER.mp4 output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4

## If Colab disconnects
Reconnect to a GPU, rerun cells 1–5, then rerun cell 6. Completed MP4s in `MyDrive/AllSortsHub-Wan2.2/generated/` are restored and skipped.

This T4 version deliberately avoids Wan GGUF and CogVideoX-5B because both caused model-loading/RAM failures in this Colab session. SVD-XT produces short image-to-video clips that can be assembled into the episode.